In [7]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [3]:
df = pd.read_parquet('../data/raw/data.parquet')

In [8]:
# EDA Básica
print("=== INFORMAÇÕES GERAIS ===")
print(f"Shape: {df.shape}")
print(f"\n{df.info()}")
print(f"\n=== ESTATÍSTICAS ===\n{df.describe()}")
print(f"\n=== VALORES FALTANTES ===\n{df.isnull().sum()}")

=== INFORMAÇÕES GERAIS ===
Shape: (1000, 11)
<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 11 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   age                               1000 non-null   int64  
 1   gender                            1000 non-null   str    
 2   daily_gaming_hours                1000 non-null   float64
 3   game_genre                        1000 non-null   str    
 4   primary_game                      1000 non-null   str    
 5   gaming_platform                   1000 non-null   str    
 6   sleep_hours                       1000 non-null   float64
 7   sleep_quality                     1000 non-null   str    
 8   sleep_disruption_frequency        1000 non-null   str    
 9   face_to_face_social_hours_weekly  1000 non-null   float64
 10  grades_gpa                        754 non-null    float64
dtypes: float64(4), int64(1), str(6)
memo

In [9]:
# Correlação
print("\n=== CORRELAÇÃO ===")
print(df.corr())

# Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', center=0)
plt.title('Matriz de Correlação')
plt.tight_layout()
plt.show()


=== CORRELAÇÃO ===


ValueError: could not convert string to float: 'Male'

In [ ]:
# Preparar dados para XGBoost - Predizendo grades_gpa
print("=== PREPARAÇÃO DE DADOS ===")

target_col = 'grades_gpa'
print(f"Predizendo: {target_col}\n")

X = df.drop(target_col, axis=1)
y = df[target_col]

print(f"Features: {X.columns.tolist()}")
print(f"Shape - X: {X.shape}, y: {y.shape}")
print(f"\nEstatísticas do Target ({target_col}):")
print(y.describe())

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"\nTrain: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
# Treinar XGBoost Regressor (para grades_gpa contínuo)
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

print("=== TREINANDO XGBOOST REGRESSOR ===")

model = XGBRegressor(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42,
    verbosity=1
)

model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=10)

# Predições
y_pred = model.predict(X_test)

# Avaliação
print("\n=== RESULTADOS ===")
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
r2 = r2_score(y_test, y_pred)

print(f"MAE (Mean Absolute Error): {mae:.4f}")
print(f"RMSE (Root Mean Squared Error): {rmse:.4f}")
print(f"R² Score: {r2:.4f}")

# Scatter plot real vs predito
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Valores Reais')
plt.ylabel('Valores Preditos')
plt.title('XGBoost - Predições vs Reais (grades_gpa)')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Feature Importance
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("\n=== FEATURE IMPORTANCE ===")
print(feature_importance)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importância')
plt.title('Feature Importance - XGBoost')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()